In [ ]:
# ==========================================
# Célda 1: Importación de librerías para ML
# ==========================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import joblib

print('Librerías de Machine Learning importadas con éxito.')

# ==========================================
# Célda 2: Carga y separación de datos (X e y)
# ==========================================
# Cargamos la misma base limpia
URL = 'https://raw.githubusercontent.com/reddyprasade/Machine-Learning/master/Heart-Disease-UCI/heart.csv'
try:
    df = pd.read_csv(URL)
except:
    from sklearn.datasets import fetch_openml
    X_openml, y_openml = fetch_openml(data_id=1574, as_frame=False, return_X_y=True)
    df = pd.DataFrame(X_openml)
    df['target'] = pd.to_numeric(y_openml)

X = df.drop(columns='target')
y = df['target']

print(f'Features (X): {X.shape}')
print(f'Target (y):   {y.shape}')

# ==========================================
# Célda 3: División Train / Test estratificada
# ==========================================
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print(f'Muestras de Entrenamiento: {X_train.shape[0]}')
print(f'Muestras de Prueba (Test): {X_test.shape[0]}')

# ==========================================
# Célda 4: Entrenamiento y Validación Cruzada
# ==========================================
pipelines = {
    'LogisticRegression': Pipeline([
        ('scaler', StandardScaler()),
        ('clf', LogisticRegression(max_iter=1000, random_state=42))
    ]),
    'DecisionTree': Pipeline([
        ('clf', DecisionTreeClassifier(max_depth=5, random_state=42))
    ]),
    'RandomForest': Pipeline([
        ('clf', RandomForestClassifier(n_estimators=200, max_depth=7, random_state=42))
    ])
}

results = {}
for name, pipe in pipelines.items():
    scores = cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy')
    results[name] = scores
    print(f'{name:20s} -> Accuracy CV: {scores.mean():.4f} ± {scores.std():.4f}')

# ==========================================
# Célda 5: Evaluación final con el mejor modelo (Random Forest o Logística)
# ==========================================
# Seleccionamos Random Forest como ejemplo de modelo final
best_model = pipelines['RandomForest']
best_model.fit(X_train, y_train)

y_pred = best_model.predict(X_test)

print("\n--- REPORTE DE CLASIFICACIÓN (TEST) ---")
print(classification_report(y_test, y_pred))

# Matriz de confusión
fig, ax = plt.subplots(figsize=(6, 5))
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, ax=ax, cmap='Blues', display_codes=['Sano', 'Enfermo'])
plt.title('Matriz de Confusión - Random Forest')
plt.show()

# ==========================================
# Célda 6: Serialización (Guardar el modelo)
# ==========================================
joblib.dump(best_model, 'modelo_heart_disease.pkl')
print('\n¡Modelo entrenado y guardado exitosamente como "modelo_heart_disease.pkl"!')